# Text classification using TF-IDF

### 1. Load the dataset from sklearn.datasets

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

In [2]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

### 2. Training data

In [3]:
twenty_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)

### 3. Test data

In [4]:
twenty_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

In [5]:
type(twenty_train)

sklearn.utils.Bunch

###  a.  You can access the values for the target variable using .target attribute 
###  b. You can access the name of the class in the target variable with .target_names


In [6]:
twenty_train.target

array([1, 1, 3, ..., 2, 2, 2])

In [7]:
twenty_train.target_names

['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

In [8]:
twenty_train.data[0:2]

['From: sd345@city.ac.uk (Michael Collier)\nSubject: Converting images to HP LaserJet III?\nNntp-Posting-Host: hampton\nOrganization: The City University\nLines: 14\n\nDoes anyone know of a good way (standard PC application/PD utility) to\nconvert tif/img/tga files into LaserJet III format.  We would also like to\ndo the same, converting to HPGL (HP plotter) files.\n\nPlease email any response.\n\nIs this the correct group?\n\nThanks in advance.  Michael.\n-- \nMichael Collier (Programmer)                 The Computer Unit,\nEmail: M.P.Collier@uk.ac.city                The City University,\nTel: 071 477-8000 x3769                      London,\nFax: 071 477-8565                            EC1V 0HB.\n',
 "From: ani@ms.uky.edu (Aniruddha B. Deglurkar)\nSubject: help: Splitting a trimming region along a mesh \nOrganization: University Of Kentucky, Dept. of Math Sciences\nLines: 28\n\n\n\n\tHi,\n\n\tI have a problem, I hope some of the 'gurus' can help me solve.\n\n\tBackground of the probl

In [9]:
type(twenty_train.data)

list

### 4.  Now with dependent and independent data available for both train and test datasets, using TfidfVectorizer fit and transform the training data and test data and get the tfidf features for both

In [10]:
# This step & the next are combined in the pipeline  below

### 5. Use logisticRegression with tfidf features as input and targets as output and train the model and report the train and test accuracy score

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
text_clf = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     ('clf', LogisticRegression()),])

text_clf = text_clf.fit(twenty_train.data, twenty_train.target)

/anaconda3/lib/python3.6/site-packages/sklearn/linear_model/logistic.py:433: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
/anaconda3/lib/python3.6/site-packages/sklearn/linear_model/logistic.py:460: FutureWarning: Default multi_class will be changed to 'auto' in 0.22. Specify the multi_class option to silence this warning.
  "this warning.", FutureWarning)


In [12]:
predicted = text_clf.predict(twenty_test.data)
np.mean(predicted == twenty_test.target)

0.8868175765645806

In [13]:
from sklearn.metrics import classification_report

In [14]:
print(classification_report(twenty_test.target,predicted))

              precision    recall  f1-score   support

           0       0.96      0.75      0.85       319
           1       0.83      0.97      0.89       389
           2       0.93      0.86      0.89       396
           3       0.87      0.94      0.90       398

   micro avg       0.89      0.89      0.89      1502
   macro avg       0.90      0.88      0.88      1502
weighted avg       0.89      0.89      0.89      1502



## Sentiment analysis <br> 

The objective of this problem is to perform Sentiment analysis from the tweets data collected from the users targeted at various mobile devices.
Based on the tweet posted by a user (text), we will classify if the sentiment of the user targeted at a particular mobile device is positive or not.

### 1. Read the dataset (tweets.csv) and drop the NA's while reading the dataset

In [15]:
tweets = pd.read_csv('tweets.csv',verbose=True,error_bad_lines=False,encoding='iso-8859-1')

Tokenization took: 10.71 ms
Type conversion took: 15.12 ms
Parser memory cleanup took: 0.01 ms


In [16]:
tweets.shape

(9093, 3)

In [17]:
tweets.dropna(inplace=True)

In [18]:
tweets.shape

(3291, 3)

In [19]:
#Remaning the columns with smaller names for ease of use
tweets.columns = ['tweet_text', 'directed','emotion']

In [20]:
tweets.head()

,tweet_text,directed,emotion
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [21]:
#Checking what possible values are in the field for emotion
tweets['emotion'].unique()

array(['Negative emotion', 'Positive emotion',
       'No emotion toward brand or product', "I can't tell"], dtype=object)

### 2. Preprocess the text and add the preprocessed text in a column with name `text` in the dataframe.

In [22]:
tweets['tweet_text'].head()

0    .@wesley83 I have a 3G iPhone. After 3 hrs twe...
1    @jessedee Know about @fludapp ? Awesome iPad/i...
2    @swonderlin Can not wait for #iPad 2 also. The...
3    @sxsw I hope this year's festival isn't as cra...
4    @sxtxstate great stuff on Fri #SXSW: Marissa M...
Name: tweet_text, dtype: object

### 3. Consider only rows having Positive emotion and Negative emotion and remove other rows from the dataframe.

In [23]:
# Slicing for rows with positive and negative emotion
Neg_bool = tweets['emotion'] == 'Negative emotion'
Pos_bool = tweets['emotion'] == 'Positive emotion'
tweets_Neg_Pos = tweets[Pos_bool | Neg_bool]
tweets_Neg_Pos.shape

(3191, 3)

### 4. Represent text as numerical data using `CountVectorizer` and get the document term frequency matrix

#### Use `vect` as the variable name for initialising CountVectorizer.

In [24]:
vect = CountVectorizer()

In [25]:
X=vect.fit_transform(tweets_Neg_Pos['tweet_text'])

In [26]:
tf = pd.DataFrame(X.toarray(),columns=vect.get_feature_names())

In [27]:
tf.head()

,000,02,03,08,10,100,100s,100tc,101,106,...,ûïmute,ûïspecials,ûïthe,ûïview,ûò,ûòand,ûó,ûójust,ûólewis,ûóthe
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 5. Find number of different words in vocabulary

In [28]:
#Number of words 
len(vect.get_feature_names())

5648

In [29]:
# Some samples of words
vect.get_feature_names()[1000:1005]

['code', 'coded', 'coders', 'coding', 'coffee']

#### Tip: To see all available functions for an Object use dir

### 6. Find out how many Positive and Negative emotions are there.

Hint: Use value_counts on that column

In [30]:
tweets_Neg_Pos['emotion'].value_counts()

Positive emotion    2672
Negative emotion     519
Name: emotion, dtype: int64

### 7. Change the labels for Positive and Negative emotions as 1 and 0 respectively and store in a different column in the same dataframe named 'Label'

Hint: use map on that column and give labels

In [31]:
tweets_Neg_Pos[tweets_Neg_Pos['emotion']=='Negative emotion'].index

Int64Index([   0,    3,   17,   38,   67,   92,  170,  172,  180,  190,
            ...
            8809, 8819, 8820, 8822, 8855, 8930, 8943, 8981, 9008, 9080],
           dtype='int64', length=519)

In [32]:
tweets_Neg_Pos.loc[tweets_Neg_Pos[tweets_Neg_Pos['emotion']=='Negative emotion'].index,'Emotion']=0
tweets_Neg_Pos.loc[tweets_Neg_Pos[tweets_Neg_Pos['emotion']=='Positive emotion'].index,'Emotion']=1
tweets_Neg_Pos['Emotion'].unique()

/anaconda3/lib/python3.6/site-packages/pandas/core/indexing.py:362: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self.obj[key] = _infer_fill_value(value)
/anaconda3/lib/python3.6/site-packages/pandas/core/indexing.py:543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self.obj[item] = s


array([0., 1.])

In [33]:
#Checking to see if the change is done
tweets_Neg_Pos['Emotion'].unique()

array([0., 1.])

### 8. Define the feature set (independent variable or X) to be `text` column and `labels` as target (or dependent variable)  and divide into train and test datasets

In [34]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(tf, tweets_Neg_Pos['Emotion'], test_size=0.33, random_state=42)
X_train.shape, y_train.shape , X_test.shape, y_test.shape 

((2137, 5648), (2137,), (1054, 5648), (1054,))

## 9. **Predicting the sentiment:**


### Use Naive Bayes and Logistic Regression and their accuracy scores for predicting the sentiment of the given text

In [35]:
logmodel_1 = LogisticRegression()
logmodel_1.fit(X_train,y_train)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='warn',
          n_jobs=None, penalty='l2', random_state=None, solver='warn',
          tol=0.0001, verbose=0, warm_start=False)

In [36]:
predictions = logmodel_1.predict(X_test)

In [37]:
print('\n\n------ Logistics Regression Classification Report -----------\n\n')
print(classification_report(y_test,predictions))



------ Logistics Regression Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.67      0.35      0.46       167
         1.0       0.89      0.97      0.93       887

   micro avg       0.87      0.87      0.87      1054
   macro avg       0.78      0.66      0.69      1054
weighted avg       0.85      0.87      0.85      1054



In [38]:
# Using Naive Bayes

nb = MultinomialNB()
nb.fit(X_train,y_train)
predictions = nb.predict(X_test)
predictions = nb.predict(X_test)
print('\n\n------ Naive Bayes Classification Report -----------\n\n')
print(classification_report(y_test,predictions))




------ Naive Bayes Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.60      0.34      0.43       167
         1.0       0.88      0.96      0.92       887

   micro avg       0.86      0.86      0.86      1054
   macro avg       0.74      0.65      0.67      1054
weighted avg       0.84      0.86      0.84      1054



## 10. Create a function called `tokenize_predict` which can take count vectorizer object as input and prints the accuracy for x (text) and y (labels)

In [39]:
def tokenize_predict(vect):
    X_tp=vect.fit_transform(tweets_Neg_Pos['tweet_text'])
    tf_tp = pd.DataFrame(X_tp.toarray(),columns=vect.get_feature_names())
    X_train_tp, X_test_tp, y_train_tp, y_test_tp = train_test_split(tf_tp, tweets_Neg_Pos['Emotion'], test_size=0.33, random_state=42)
    nb = MultinomialNB()
    nb.fit(X_train_tp, y_train_tp)
    y_pred_class_tp = nb.predict(X_test_tp)
    print('\n\n------ Accuracy -----------\n\n')
    print('Accuracy: ', metrics.accuracy_score(y_test_tp, y_pred_class_tp))
    print('\n\n------ Classification Report -----------\n\n')
    print(classification_report(y_test_tp,y_pred_class_tp))

In [40]:
vect = CountVectorizer()
tokenize_predict(vect)



------ Accuracy -----------


Accuracy:  0.8586337760910816


------ Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.60      0.34      0.43       167
         1.0       0.88      0.96      0.92       887

   micro avg       0.86      0.86      0.86      1054
   macro avg       0.74      0.65      0.67      1054
weighted avg       0.84      0.86      0.84      1054



### Create a count vectorizer function which includes n_grams = 1,2  and pass it to tokenize_predict function to print the accuracy score

In [41]:
vect_ng = CountVectorizer(ngram_range=(1,2))
tokenize_predict(vect_ng)



------ Accuracy -----------


Accuracy:  0.8415559772296015


------ Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.50      0.44      0.47       167
         1.0       0.90      0.92      0.91       887

   micro avg       0.84      0.84      0.84      1054
   macro avg       0.70      0.68      0.69      1054
weighted avg       0.83      0.84      0.84      1054



### Create a count vectorizer function with stopwords = 'english'  and pass it to tokenize_predict function to print the accuracy score

In [42]:
vect_st = CountVectorizer(stop_words='english')
tokenize_predict(vect_st)



------ Accuracy -----------


Accuracy:  0.8510436432637571


------ Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.54      0.37      0.44       167
         1.0       0.89      0.94      0.91       887

   micro avg       0.85      0.85      0.85      1054
   macro avg       0.72      0.65      0.68      1054
weighted avg       0.83      0.85      0.84      1054



### Create a count vectorizer function with stopwords = 'english' and max_features =300  and pass it to tokenize_predict function to print the accuracy score

In [43]:
vect_st_mf = CountVectorizer(stop_words='english',max_features=300)
tokenize_predict(vect_st_mf)



------ Accuracy -----------


Accuracy:  0.8263757115749526


------ Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.44      0.34      0.38       167
         1.0       0.88      0.92      0.90       887

   micro avg       0.83      0.83      0.83      1054
   macro avg       0.66      0.63      0.64      1054
weighted avg       0.81      0.83      0.82      1054



### Create a count vectorizer function with n_grams = 1,2  and max_features = 15000  and pass it to tokenize_predict function to print the accuracy score

In [44]:
vect_ng_mf = CountVectorizer(ngram_range=(1,2),max_features=15000)
tokenize_predict(vect_ng_mf)



------ Accuracy -----------


Accuracy:  0.8500948766603416


------ Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.53      0.46      0.49       167
         1.0       0.90      0.92      0.91       887

   micro avg       0.85      0.85      0.85      1054
   macro avg       0.72      0.69      0.70      1054
weighted avg       0.84      0.85      0.85      1054



### Create a count vectorizer function with n_grams = 1,2  and include terms that appear at least 2 times (min_df = 2)  and pass it to tokenize_predict function to print the accuracy score

In [45]:
vect_ng_mdf = CountVectorizer(ngram_range=(1,2),min_df=2)
tokenize_predict(vect_ng_mdf)



------ Accuracy -----------


Accuracy:  0.8462998102466793


------ Classification Report -----------


              precision    recall  f1-score   support

         0.0       0.52      0.47      0.49       167
         1.0       0.90      0.92      0.91       887

   micro avg       0.85      0.85      0.85      1054
   macro avg       0.71      0.69      0.70      1054
weighted avg       0.84      0.85      0.84      1054

